# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
#WEEEK2 - Day2






# ============================================================
# IMPORTS
# ============================================================

import os
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_contents

import gradio as gr


# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================
# Load the API keys and other environment variables from
# the .env file.
#
# Using a .env file keeps sensitive API keys out of the
# Python source code.

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")


# ============================================================
# CHECK API KEYS
# ============================================================
# Check whether our API keys were successfully loaded.
#
# We only print the beginning of each key for debugging.
# NEVER print your complete API key.

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")


# ============================================================
# CONNECT TO OPENAI
# ============================================================
# Create an OpenAI client.
#
# We can use the OpenAI Python library to communicate with
# OpenAI's API.

openai = OpenAI()


# ============================================================
# CONNECT TO GOOGLE GEMINI
# ============================================================
# Google provides an OpenAI-compatible API endpoint.
#
# Because Gemini supports this interface, we can use the
# OpenAI Python client while changing the base URL to
# Google's Gemini endpoint.

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"


# Create the Gemini client using our Google API key.

gemini = OpenAI(
    api_key=google_api_key,
    base_url=gemini_url
)


# ============================================================
# GEMINI MODEL
# ============================================================
# Store the model name in a variable.
#
# This makes it easier to change models later without
# having to search through the entire program.

MODEL_GEMINI = "gemini-3-flash-preview"


# ============================================================
# SYSTEM MESSAGE
# ============================================================
# The system message controls how Gemini should behave
# when responding to the user.

system_message = "You are a helpful assistant."


# ============================================================
# GEMINI MESSAGE FUNCTION
# ============================================================
# This function takes a user's prompt and sends it to Gemini.
#
# We create two messages:
#
# 1. system - tells Gemini how it should behave
# 2. user   - contains the user's actual question
#
# The function then returns Gemini's response as text.

def message_gemini(prompt):

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # Send the messages to the Gemini API.

    response = gemini.chat.completions.create(
        model=MODEL_GEMINI,
        messages=messages
    )

    # Return only the text generated by Gemini.

    return response.choices[0].message.content


# ============================================================
# SIMPLE FUNCTION EXAMPLE
# ============================================================
# This was an earlier example used to learn how functions
# work with Gradio.
#
# It takes text and converts it to uppercase.

def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()


# Test the shout function.

print(shout("hello"))


# ============================================================
# BASIC GRADIO INTERFACE
# ============================================================
# This is an earlier Gradio example.
#
# It has been commented out because we are now using Gemini
# instead of the simple shout function.
#
# The auth parameter can be used to add username/password
# authentication to a Gradio application.

# gr.Interface(
#     fn=shout,
#     inputs="textbox",
#     outputs="textbox",
#     flagging_mode="never"
# ).launch(
#     inbrowser=True,
#     auth=("aidan", "aidan13@")
# )


# ============================================================
# FORCE DARK MODE IN GRADIO
# ============================================================
# JavaScript can be passed to Gradio to customize the
# behaviour of the web interface.
#
# This JavaScript checks the URL for the dark theme and
# redirects the page to the dark theme if necessary.

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);

    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""


# Earlier dark-mode Gradio example.
# Commented out because we are using the newer interface below.

# gr.Interface(
#     fn=shout,
#     inputs="textbox",
#     outputs="textbox",
#     flagging_mode="never",
#     js=force_dark_mode
# ).launch()


# ============================================================
# EARLIER GEMINI GRADIO INTERFACE
# ============================================================
# This was an earlier version of the Gemini interface.
#
# It is kept here so we can see the progression of the
# application as we learn more about Gradio.

# message_input = gr.Textbox(
#     label="Your message",
#     info="Ask Gemini anything",
#     lines=7
# )

# message_output = gr.Textbox(
#     label="Gemini Response",
#     lines=8
# )


# view = gr.Interface(
#     fn=message_gemini,
#     title="Gemini Assistant",
#     inputs=[message_input],
#     outputs=[message_output],
#     examples=[
#         "Explain APIs to me",
#         "What is Python?",
#         "Explain machine learning simply"
#     ],
#     flagging_mode="never"
# )

# view.launch()


# ============================================================
# MARKDOWN GEMINI INTERFACE
# ============================================================
# We now tell Gemini to respond using Markdown.
#
# Markdown allows the response to be displayed with
# formatting such as:
#
# - Headings
# - Bullet points
# - Bold text
# - Lists
# - Other Markdown formatting

system_message = (
    "You are a helpful assistant that responds in markdown "
    "without code blocks"
)


message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for Gemini",
    lines=7
)


message_output = gr.Markdown(
    label="Response"
)


view = gr.Interface(
    fn=message_gemini,
    title="GEMINI",
    inputs=message_input,
    outputs=message_output,
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI Engineer",
    ],
    flagging_mode="never"
)


# ============================================================
# STREAMING GEMINI RESPONSES
# ============================================================
# Instead of waiting for Gemini to generate the entire
# response before displaying anything, streaming allows
# us to receive the response piece by piece.
#
# The `yield` keyword allows this function to return
# partial results as Gemini generates them.


def stream_gemini(prompt):

    # Create the messages that will be sent to Gemini.

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # stream=True tells Gemini to send the response
    # in chunks instead of waiting for the full response.

    stream = gemini.chat.completions.create(
        model=MODEL_GEMINI,
        messages=messages,
        stream=True
    )

   

    result = ""

    

    for chunk in stream:

        # Add the new piece of text to our existing result.
        #
        # Some chunks may not contain any text, which is
        # why we use `or ""`.

        result += chunk.choices[0].delta.content or ""

        # yield sends the current result back to the caller
        # without ending the function.
       

        yield result


OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ.Ab8RN
Shout has been called with input hello
HELLO


In [ ]:
#Week2 - Day2 Exercise



# ============================================================
# COMPANY BROCHURE GENERATOR
# ============================================================
# This application takes a company's name and website URL,
# retrieves information from the website, and uses Gemini
# to generate a short company brochure.
#
# The brochure is designed for:
# - Prospective customers
# - Investors
# - Potential employees / recruits
#
# Gemini will return the brochure in Markdown format.


# ============================================================
# IMPORTS
# ============================================================

import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI

import gradio as gr


# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================
# Load the Google API key from the .env file.

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")


# Check that the API key was loaded successfully.

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")


# ============================================================
# CONNECT TO GOOGLE GEMINI
# ============================================================
# Gemini provides an OpenAI-compatible API endpoint.
#
# This means we can use the OpenAI Python library to
# communicate with Gemini.

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(
    api_key=google_api_key,
    base_url=gemini_url
)


# ============================================================
# GEMINI MODEL
# ============================================================
# Store the model name in a variable so it can easily
# be changed later.

MODEL_GEMINI = "gemini-3-flash-preview"


# ============================================================
# SYSTEM MESSAGE
# ============================================================
# Tell Gemini exactly what its job is.
#
# The system message instructs Gemini to analyze the
# company's website and create a brochure.

system_message = """
You are an assistant that analyzes the contents of a company
website landing page and creates a short brochure about the
company for prospective customers, investors and recruits.

Respond in markdown without code blocks.
"""


# ============================================================
# FETCH WEBSITE CONTENT
# ============================================================
# This function downloads the company's website and extracts
# the useful text from the HTML.
#
# We remove elements such as scripts and styles because they
# aren't useful when creating the brochure.

def fetch_website_contents(url):

    # Send a request to the website.

    response = requests.get(
        url,
        timeout=10,
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    # Raise an error if the website could not be accessed.

    response.raise_for_status()

    # Parse the HTML returned by the website.

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # Remove unnecessary HTML elements.

    for element in soup([
        "script",
        "style",
        "nav",
        "footer"
    ]):
        element.decompose()

    # Extract the remaining visible text.

    text = soup.get_text(
        separator=" ",
        strip=True
    )

    return text


# ============================================================
# STREAM BROCHURE
# ============================================================
# This function:
#
# 1. Receives the company name
# 2. Receives the company's website URL
# 3. Fetches the website content
# 4. Sends the information to Gemini
# 5. Streams the brochure back to Gradio


def stream_brochure(company_name, url):

    # Start with an empty response.

    yield ""

    try:

        # Create the prompt for Gemini.

        prompt = (
            f"Please generate a company brochure for "
            f"{company_name}.\n\n"
            f"Here is their landing page content:\n\n"
        )

        # Fetch the company's website.

        website_content = fetch_website_contents(url)

        # Add the website content to the prompt.

        prompt += website_content

        # Ask Gemini to generate the brochure.

        stream = gemini.chat.completions.create(
            model=MODEL_GEMINI,
            messages=[
                {
                    "role": "system",
                    "content": system_message
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            stream=True
        )

        # Keep track of the response as it arrives.

        result = ""

        # Process each chunk from Gemini.

        for chunk in stream:

            # Extract the text from the current chunk.

            result += chunk.choices[0].delta.content or ""

            # Send the updated response to Gradio.

            yield result

    except Exception as e:

        # Display a helpful error message if something
        # goes wrong while fetching the website or calling
        # the Gemini API.

        yield f"### Error\n\n`{str(e)}`"


# ============================================================
# GRADIO INPUTS
# ============================================================
# Create the fields that the user will interact with.

name_input = gr.Textbox(
    label="Company name:",
    placeholder="e.g. Google"
)


url_input = gr.Textbox(
    label="Landing page URL:",
    placeholder="https://example.com"
)


# ============================================================
# GRADIO OUTPUT
# ============================================================
# Markdown allows the generated brochure to display
# headings, bullet points, bold text and other formatting.

message_output = gr.Markdown(
    label="Company Brochure:"
)


# ============================================================
# GRADIO INTERFACE
# ============================================================
# Connect our stream_brochure() function to Gradio.

view = gr.Interface(
    fn=stream_brochure,
    title="AI Company Brochure Generator",
    description=(
        "Enter a company name and website URL. "
        "Gemini will analyze the website and generate "
        "a short company brochure."
    ),
    inputs=[
        name_input,
        url_input
    ],
    outputs=[
        message_output
    ],
    examples=[
        [
            "Hugging Face",
            "https://huggingface.co"
        ],
        [
            "Google",
            "https://google.com"
        ]
    ],
    flagging_mode="never"
)


# ============================================================
# LAUNCH APPLICATION
# ============================================================

view.launch()


In [ ]:
#Week 2 - Day 3 (Building ChatUI with Gradio)

##Imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

#Load env variables

load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')

if google_api_key:
    print(f"Google API Keys exist and begins with {google_api_key[:8]}")
else:
    print("Google API key not set")  


#Initailise
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(
    api_key = google_api_key,
    base_url = gemini_url
)

MODEL = "gemini-3-flash-preview"

# system_message = "You are a helpfull assistant"


#CallBack

# def chat(message,history):
#     return "Piesan"

# gr.ChatInterface(fn=chat, type="messages").launch()

#Better callback

# def chat(message, history):
#     history = [{"role": h["role"], "content": h["content"]} for h in history]
#     messages = [{"role":"system", "content":system_message}]+ history + [{"role": "user", "content":message}]

    

#     response = gemini.chat.completions.create(
#         model=MODEL,
#         messages=messages
#     )

#     return(response.choices[0].message.content)

#Adding stream

#     stream = gemini.chat.completions.create(model=MODEL, messages=messages, stream=True)
#     response= " "
#     for chunk in stream:
#         reponse += chunk.choices[0].delta.content or ''
#         yield response

# gr.ChatInterface(fn=chat, type="messages").launch()


system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = gemini.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

gr.ChatInterface(fn=chat, type="messages").launch()





In [ ]:
## Day 4
# Project Airline AI Assistant

# ============================================================
# IMPORTS
# ============================================================

import os
import json
import sqlite3

from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_contents

import gradio as gr


# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")


# ============================================================
# CONNECT TO GOOGLE GEMINI
# ============================================================

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(
    api_key=google_api_key,
    base_url=gemini_url
)


# ============================================================
# GEMINI MODEL
# ============================================================

MODEL_GEMINI = "gemini-3-flash-preview"


# ============================================================
# SYSTEM MESSAGE
# ============================================================

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""


# ============================================================
# TICKET PRICE DATABASE
# ============================================================

# SQLite database file.

DB = "prices.db"


# Create the database and prices table if they don't exist.

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS prices (
            city TEXT PRIMARY KEY,
            price REAL
        )
    """)

    conn.commit()


# ============================================================
# DATABASE TOOL - GET TICKET PRICE
# ============================================================

def get_ticket_price(city):

    print(
        f"DATABASE TOOL CALLED: Getting price for {city}",
        flush=True
    )

    with sqlite3.connect(DB) as conn:

        cursor = conn.cursor()

        cursor.execute(
            "SELECT price FROM prices WHERE city = ?",
            (city.lower(),)
        )

        result = cursor.fetchone()

        if result:
            return f"Ticket price to {city} is ${result[0]}"

        return "No price data available for this city"


# ============================================================
# DATABASE TOOL - SET TICKET PRICE
# ============================================================

def set_ticket_price(city, price):

    with sqlite3.connect(DB) as conn:

        cursor = conn.cursor()

        cursor.execute(
            """
            INSERT INTO prices (city, price)
            VALUES (?, ?)
            ON CONFLICT(city)
            DO UPDATE SET price = ?
            """,
            (city.lower(), price, price)
        )

        conn.commit()


# ============================================================
# ADD SAMPLE TICKET PRICES
# ============================================================

ticket_prices = {
    "london": 799,
    "paris": 899,
    "tokyo": 1420,
    "sydney": 2999
}


for city, price in ticket_prices.items():
    set_ticket_price(city, price)


# Test the database tool.

print(get_ticket_price("London"))


# ============================================================
# DEFINE THE TICKET PRICE FUNCTION FOR GEMINI
# ============================================================

price_function = {
    "name": "get_ticket_price",

    "description": (
        "Get the price of a return ticket to the destination city."
    ),

    "parameters": {
        "type": "object",

        "properties": {
            "destination_city": {
                "type": "string",
                "description": (
                    "The city that the customer wants to travel to"
                ),
            },
        },

        "required": ["destination_city"],

        "additionalProperties": False
    }
}


# Tell Gemini which tools it is allowed to use.

tools = [
    {
        "type": "function",
        "function": price_function
    }
]


# ============================================================
# HANDLE TOOL CALLS
# ============================================================

def handle_tool_call(message):

    tool_call = message.tool_calls[0]

    if tool_call.function.name == "get_ticket_price":

        # Convert Gemini's JSON arguments into a Python dictionary.

        arguments = json.loads(
            tool_call.function.arguments
        )

        city = arguments.get("destination_city")

        # Call our actual Python database function.

        price_details = get_ticket_price(city)

        # Return the tool result to Gemini.

        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }

        return response


# ============================================================
# CHAT FUNCTION
# ============================================================

def chat(message, history):

    # Convert Gradio history into the format Gemini expects.

    history = [
        {
            "role": h["role"],
            "content": h["content"]
        }
        for h in history
    ]


    # Build the complete conversation.

    messages = [
        {
            "role": "system",
            "content": system_message
        }
    ] + history + [
        {
            "role": "user",
            "content": message
        }
    ]


    # ========================================================
    # FIRST GEMINI CALL
    # ========================================================
    # Gemini decides whether it needs to use our ticket
    # price tool.

    response = gemini.chat.completions.create(
        model=MODEL_GEMINI,
        messages=messages,
        tools=tools
    )


    # ========================================================
    # CHECK FOR TOOL CALL
    # ========================================================

    if response.choices[0].finish_reason == "tool_calls":

        # Get Gemini's requested tool call.

        assistant_message = response.choices[0].message

        # Execute our Python database function.

        tool_response = handle_tool_call(
            assistant_message
        )

        # Add Gemini's tool request to the conversation.

        messages.append(assistant_message)

        # Add the database result.

        messages.append(tool_response)


        # ====================================================
        # SECOND GEMINI CALL
        # ====================================================
        # Gemini now receives the database result and can
        # formulate the final answer to the customer.

        response = gemini.chat.completions.create(
            model=MODEL_GEMINI,
            messages=messages
        )


    # Return Gemini's final answer.

    return response.choices[0].message.content


# ============================================================
# GRADIO CHAT INTERFACE
# ============================================================

gr.ChatInterface(
    fn=chat,
    type="messages"
).launch()

Google API Key exists and begins AQ.Ab8RN
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


NameError: name 'get_tickets_price' is not defined